# YOLOV11 — Benchmark on OOD Dataset (22 classes)


In [1]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.6.0+cu124
CUDA: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [2]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov11n", "yolov11s", "yolov11m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../dataset/data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov11.csv"
PERCLASS_CSV = "benchmark_yolov11_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov11n.pt",
    "yolov11s.pt",
    "yolov11m.pt",
]

In [3]:
def benchmark_model(model_name):
    """Train a model on OOD and return benchmark metrics."""
    print(f"\n{'='*60}")
    print(f"  BENCHMARK: {model_name}")
    print(f"{'='*60}")

    resume_path = RESUME_CKPT
    use_resume = (
        resume_path
        and RESUME_MODEL == model_name
        and Path(resume_path).is_file()
    )
    if use_resume:
        print(f"  Resuming from {resume_path}")
        model = YOLO(resume_path)
        model.train(resume=True)
    else:
        if resume_path and RESUME_MODEL == model_name:
            print(f"  (RESUME_CKPT missing, training from scratch: {resume_path})")
        model = YOLO(model_name)
        model.train(
            data=DATA_YAML,
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH,
            device=DEVICE,
            workers=WORKERS,
            amp=AMP,
            name=f"{model_name.replace('.pt', '')}_ood",
            patience=PATIENCE,
            save=True,
            plots=True,
        )

    best_path = Path(model.trainer.best)
    best_model = YOLO(str(best_path))

    metrics = best_model.val(
        data=DATA_YAML, split="test", device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS
    )

    test_img_dir = TEST_IMG_DIR
    _ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    test_images = [p for p in test_img_dir.glob("*.*") if p.suffix.lower() in _ext][:200]
    if not test_images:
        raise FileNotFoundError(f"No test images under {test_img_dir.resolve()}")
    latencies = []
    for _ in range(3):
        best_model(str(test_images[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    for img_path in test_images:
        t0 = time.perf_counter()
        best_model(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)

    size_mb = best_path.stat().st_size / 1e6 if best_path.exists() else 0.0
    params_m = sum(p.numel() for p in best_model.model.parameters()) / 1e6
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0

    row = {
        "model":         model_name.replace(".pt", ""),
        "mAP@0.5":       round(float(metrics.box.map50), 4),
        "mAP@0.5:0.95":  round(float(metrics.box.map), 4),
        "precision":     round(p, 4),
        "recall":        round(r, 4),
        "F1":            round(f1, 4),
        "speed_ms/img":  round(float(np.mean(latencies)), 2),
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }

    per_class = {}
    ap50_per_class = metrics.box.ap50
    for i, name in enumerate(CLASS_NAMES):
        if i < len(ap50_per_class):
            per_class[name] = round(float(ap50_per_class[i]), 4)

    del model, best_model
    gc.collect()
    _safe_cuda_empty_cache()

    return row, per_class


def _runs_detect_dir() -> Path:
    """``runs/detect`` whether the notebook cwd is ``benchmark/`` or project root."""
    for rel in ("../runs/detect", "runs/detect"):
        p = Path(rel).resolve()
        if p.is_dir():
            return p
    return Path("../runs/detect").resolve()


def _find_newest_best_pt(variant: str):
    """Newest weights for this variant under runs/detect.

    Matches ``yolo11n``, ``yolo11n_ood``, ``yolo11n_ood2``, etc.
    """
    root = _runs_detect_dir()
    if not root.is_dir():
        return None
    cands = []
    for w in root.glob("**/weights/best.pt"):
        run_name = w.parent.parent.name
        if run_name == variant or run_name.startswith(f"{variant}_"):
            cands.append(w)
    return max(cands, key=lambda p: p.stat().st_mtime) if cands else None


def metrics_from_weights(best_pt: Path, model_key: str):
    """Test-set metrics + speed from a saved best.pt (no training)."""
    print(f"\n{'='*60}\n  METRICS ONLY: {model_key}\n  weights: {best_pt}\n{'='*60}")
    best_pt = best_pt.resolve()
    best_model = YOLO(str(best_pt))
    metrics = best_model.val(
        data=DATA_YAML, split="test", device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS
    )
    test_img_dir = TEST_IMG_DIR
    _ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
    test_images = [p for p in test_img_dir.glob("*.*") if p.suffix.lower() in _ext][:200]
    if not test_images:
        raise FileNotFoundError(f"No test images under {test_img_dir.resolve()}")
    latencies = []
    for _ in range(3):
        best_model(str(test_images[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    for img_path in test_images:
        t0 = time.perf_counter()
        best_model(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)
    size_mb = best_pt.stat().st_size / 1e6 if best_pt.exists() else 0.0
    params_m = sum(p.numel() for p in best_model.model.parameters()) / 1e6
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    row = {
        "model":         model_key,
        "mAP@0.5":       round(float(metrics.box.map50), 4),
        "mAP@0.5:0.95":  round(float(metrics.box.map), 4),
        "precision":     round(p, 4),
        "recall":        round(r, 4),
        "F1":            round(f1, 4),
        "speed_ms/img":  round(float(np.mean(latencies)), 2),
        "size_MB":       round(size_mb, 1),
        "params_M":      round(params_m, 1),
    }
    per_class = {}
    ap50_per_class = metrics.box.ap50
    for i, name in enumerate(CLASS_NAMES):
        if i < len(ap50_per_class):
            per_class[name] = round(float(ap50_per_class[i]), 4)
    del best_model
    gc.collect()
    _safe_cuda_empty_cache()
    return row, per_class

In [4]:
rows = []
all_per_class = {}

if RUN_TRAINING:
    print("RUN_TRAINING=True — full training for each entry in MODELS.\n")
    for model_name in MODELS:
        try:
            row, per_class = benchmark_model(model_name)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  mAP@0.5       : {row['mAP@0.5']}")
            print(f"  mAP@0.5:0.95  : {row['mAP@0.5:0.95']}")
            print(f"  Precision     : {row['precision']}")
            print(f"  Recall        : {row['recall']}")
            print(f"  F1            : {row['F1']}")
            print(f"  Speed         : {row['speed_ms/img']} ms/img")
            print(f"  Size          : {row['size_MB']} MB")
            print(f"  Params        : {row['params_M']} M")
        except Exception as e:
            print(f"  SKIPPED {model_name}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()
            print("  If CUDA errors persist: restart the kernel, set WORKERS=0 or AMP=False in the config cell, then rerun.")
else:
    print("RUN_TRAINING=False — building table from existing best.pt (no training).\n")
    print(f"Resolved runs/detect -> {_runs_detect_dir()}\n")
    for variant in BENCHMARK_VARIANTS:
        wp = _find_newest_best_pt(variant)
        if wp is None:
            print(f"  Skip {variant}: no matching best.pt under {_runs_detect_dir()} (expect folder '{variant}' or '{variant}_*')")
            continue
        try:
            row, per_class = metrics_from_weights(wp, variant)
            rows.append(row)
            all_per_class[row["model"]] = per_class
            print(f"\n  {variant}: mAP@0.5={row['mAP@0.5']}  speed={row['speed_ms/img']} ms/img")
        except Exception as e:
            print(f"  SKIPPED {variant}: {e}")
            gc.collect()
            _safe_cuda_empty_cache()

_cols = [
    "model", "mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1",
    "speed_ms/img", "size_MB", "params_M",
]
df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)
df.to_csv(RESULTS_CSV, index=False)

if all_per_class:
    df_pc = pd.DataFrame(all_per_class).T
else:
    df_pc = pd.DataFrame(columns=CLASS_NAMES)
df_pc.index.name = "model"
df_pc.to_csv(PERCLASS_CSV)

print(f"\nSaved -> {RESULTS_CSV} ({len(df)} row(s))")
print(f"Saved -> {PERCLASS_CSV}")

RUN_TRAINING=False — building table from existing best.pt (no training).

Resolved runs/detect -> C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect


  METRICS ONLY: yolo11n
  weights: C:\Users\admin\PFE\PFE-Obstacle-Detection\runs\detect\yolo11n_ood\weights\best.pt
Ultralytics 8.4.30  Python-3.11.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,586,442 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1620.4797.2 MB/s, size: 129.9 KB)
val: Scanning C:\Users\admin\PFE\PFE-Obstacle-Detection\dataset\test\labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 7.9it/s 8.0s0.1s
                   all       1000       2945      0.688      0.581      0.629      0.447
                 bench         67        115      0.629      0.496       0.57    

In [5]:
import pandas as pd
from pathlib import Path
from IPython.display import display

RESULTS_CSV = "benchmark_yolov11_ood.csv"

_csv = Path(RESULTS_CSV)
if not _csv.is_file() or _csv.stat().st_size < 5:
    print("No results CSV yet. Run the **build benchmark** cell (config → functions → build CSV).")
else:
    df = pd.read_csv(RESULTS_CSV)
    df.columns = [str(c).strip().lstrip("\ufeff") for c in df.columns]
    if df.empty or "model" not in df.columns:
        print("benchmark_yolov11_ood.csv has no data rows yet. Run the build-benchmark cell after training or evaluating best.pt.")
        print("Columns:", list(df.columns))
    else:
        print("=" * 60)
        print("  YOLOv11 BENCHMARK — Indoor Dataset (22 classes)")
        print("=" * 60)
        print("\n", df.to_string(index=False), "\n")
        fmt = {}
        for c in df.columns:
            if c == "speed_ms/img":
                fmt[c] = "{:.1f} ms"
            elif c == "size_MB":
                fmt[c] = "{:.1f} MB"
            elif c == "params_M":
                fmt[c] = "{:.1f} M"
            elif c in ("mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1"):
                fmt[c] = "{:.4f}"
        hi_max = [c for c in ["mAP@0.5", "mAP@0.5:0.95", "precision", "recall", "F1"] if c in df.columns]
        hi_min = [c for c in ["speed_ms/img", "size_MB", "params_M"] if c in df.columns]
        styled = df.style.set_caption("YOLOv11 Benchmark — Indoor Dataset").format(fmt, na_rep="—")
        if hi_max:
            styled = styled.highlight_max(subset=hi_max, color="#2d6a2e")
        if hi_min:
            styled = styled.highlight_min(subset=hi_min, color="#1a5276")
        styled = (
            styled
            .set_properties(**{"text-align": "center", "font-size": "13px"})
            .set_table_styles([
                {"selector": "caption", "props": [("font-size", "16px"), ("font-weight", "bold"), ("padding", "10px")]},
                {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("padding", "8px")]},
            ])
            .hide(axis="index")
        )
        display(styled)

  YOLOv11 BENCHMARK — OOD Dataset (22 classes)

   model  mAP@0.5  mAP@0.5:0.95  precision  recall     F1  speed_ms/img  size_MB  params_M
yolo11n   0.6289        0.4471     0.6883  0.5810 0.6301         15.47      5.5       2.6
yolo11s   0.6292        0.4480     0.6882  0.5859 0.6329         18.80     19.2       9.4
yolo11m   0.6324        0.4515     0.6772  0.5985 0.6354         29.75     40.5      20.0 



model,mAP@0.5,mAP@0.5:0.95,precision,recall,F1,speed_ms/img,size_MB,params_M
yolo11n,0.6289,0.4471,0.6883,0.5810,0.6301,15.5 ms,5.5 MB,2.6 M
yolo11s,0.6292,0.4480,0.6882,0.5859,0.6329,18.8 ms,19.2 MB,9.4 M
yolo11m,0.6324,0.4515,0.6772,0.5985,0.6354,29.8 ms,40.5 MB,20.0 M


In [6]:
import pandas as pd
from pathlib import Path
from IPython.display import display

PERCLASS_CSV = "benchmark_yolov11_perclass.csv"

_pcsv = Path(PERCLASS_CSV)
if not _pcsv.is_file() or _pcsv.stat().st_size < 5:
    print("No per-class CSV yet. Run the **build benchmark** cell first.")
else:
    df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)
    if df_pc.empty:
        print("Per-class table is empty.")
    else:
        print("Per-class mAP@0.5 across YOLOv11 variants")
        print("-" * 60)
        styled_pc = (
            df_pc.style
            .set_caption("Per-Class mAP@0.5 — YOLOv11 Benchmark")
            .format("{:.4f}")
            .background_gradient(cmap="RdYlGn", axis=None, vmin=0, vmax=1)
            .set_properties(**{"text-align": "center", "font-size": "12px"})
            .set_table_styles([
                {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold")]},
                {"selector": "th", "props": [("background-color", "#1a1a2e"), ("color", "white"), ("font-size", "11px"), ("padding", "6px")]},
            ])
        )
        display(styled_pc)

Per-class mAP@0.5 across YOLOv11 variants
------------------------------------------------------------


,bench,bicycle,bus,bus_stop,car,crutch,curb,dog,fire_hydrant,motorcycle,person,pole,spherical_roadblock,stairs,stop_sign,street_light,traffic_light,train,tree,truck,warning_column,waste_container
model,,,,,,,,,,,,,,,,,,,,,,
yolo11n,0.5698,0.5797,0.7548,0.9606,0.6523,0.5146,0.5282,0.6906,0.8293,0.6879,0.2067,0.3124,0.9153,0.6310,0.9111,0.2476,0.4649,0.8067,0.2075,0.6545,0.8701,0.8395
yolo11s,0.5691,0.5313,0.7596,0.9716,0.6541,0.6512,0.4167,0.7200,0.8663,0.6251,0.2124,0.3375,0.9301,0.6150,0.9361,0.2860,0.5281,0.7431,0.1709,0.6141,0.8615,0.8423
yolo11m,0.6030,0.5681,0.7627,0.9629,0.6618,0.5536,0.4337,0.7125,0.8233,0.7003,0.2481,0.3384,0.9182,0.6267,0.9258,0.2893,0.4299,0.7932,0.1842,0.6561,0.8593,0.8615


In [7]:
import matplotlib.pyplot as plt
from pathlib import Path

_csv = Path(RESULTS_CSV)
if not _csv.is_file() or _csv.stat().st_size < 5:
    print("No results CSV — run the build-benchmark cell first.")
else:
    df = pd.read_csv(RESULTS_CSV)
    if df.empty or "model" not in df.columns:
        print("Results CSV has no data rows — skip chart.")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].barh(df["model"], df["mAP@0.5"], color="#2d6a2e")
        axes[0].set_xlabel("mAP@0.5")
        axes[0].set_title("Accuracy (mAP@0.5)")
        axes[0].set_xlim(0, 1)
        axes[1].barh(df["model"], df["speed_ms/img"], color="#1a5276")
        axes[1].set_xlabel("ms / image")
        axes[1].set_title("Inference Speed")
        axes[2].barh(df["model"], df["size_MB"], color="#c0392b")
        axes[2].set_xlabel("MB")
        axes[2].set_title("Model Size")
        plt.suptitle("YOLOv11 Benchmark — Indoor Dataset", fontsize=16, fontweight="bold")
        plt.tight_layout()
        plt.savefig("benchmark_yolov11_chart.png", dpi=150, bbox_inches="tight")
        plt.show()

<Figure size 1800x500 with 3 Axes>

In [8]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os

# Only evaluate saved weights for tables by default.
RUN_TRAINING = False
BENCHMARK_VARIANTS = ["yolov11n", "yolov11s", "yolov11m"]

WORKERS = 2
AMP = True
RESUME_CKPT = None
RESUME_MODEL = None

DATA_YAML    = "../data.yaml"
IMG_SIZE     = 640
EPOCHS       = 70
BATCH        = 16
DEVICE       = 0
PATIENCE     = 20
RESULTS_CSV  = "benchmark_yolov11.csv"
PERCLASS_CSV = "benchmark_yolov11_perclass.csv"

CLASS_NAMES = ["bench", "bicycle", "bus", "bus_stop", "car", "crutch", "curb", "dog", "fire_hydrant", "motorcycle", "person", "pole", "spherical_roadblock", "stairs", "stop_sign", "street_light", "traffic_light", "train", "tree", "truck", "warning_column", "waste_container"]

MODELS = [
    "yolov11n.pt",
    "yolov11s.pt",
    "yolov11m.pt",
]

Best model: yolo11m — loading ..\runs\detect\yolo11m_ood\weights\best.pt


<Figure size 1800x1800 with 9 Axes>